In [9]:
import pandas as pd
from pathlib import Path

raw_path = Path("D:/python projects/Mini Retail Sales Pipeline/data/raw")

customers_path = raw_path / "customers.csv"
orders_path = raw_path / "orders.csv"
products_path = raw_path / "products.csv"

customers = pd.read_csv(customers_path)
orders = pd.read_csv(orders_path)
products = pd.read_csv(products_path)

In [30]:
# creating a function which will clean customers data


# a cleanning function for customers
def clean_customers(customers):
    customers["city"] = customers["city"].fillna("Unknown")
    customers = customers.drop_duplicates(subset=["customer_id"], keep="first")
    customers = customers.dropna(subset=["customer_name"])

    return customers
# a cleanning function for products
def clean_products(products):
    products = products.dropna(subset="product_id")
    products["category"] = products["category"].fillna("Uncategorized")
    products["unit_cost"] = products["unit_cost"].astype("float")
    products["unit_price"] = products["unit_price"].astype("float")

    return products
# a cleanning function for orders
def clean_orders(orders):
    orders = orders.drop_duplicates(subset="order_id", keep="first")
    orders = orders.dropna(subset="quantity")
    orders = orders[ orders["quantity"] > 0 ]
    orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
    orders["discount_percent"] = orders["discount_percent"].astype("float")

    return orders

    
clean_path = Path("D:\python projects\Mini Retail Sales Pipeline\data\processed")
clean_path.mkdir(parents=True, exist_ok=True)

customers_clean = clean_customers(customers)
products_clean = clean_products(products)
orders_clean = clean_orders(orders)

customers_clean.to_csv(clean_path / "customers_clean.csv", index=False)
products_clean.to_csv(clean_path / "products_clean.csv", index=False)
orders_clean.to_csv(clean_path / "orders_clean.csv", index=False)

<>:30: SyntaxWarning: invalid escape sequence '\p'
<>:30: SyntaxWarning: invalid escape sequence '\p'
C:\Users\User\AppData\Local\Temp\ipykernel_10036\1778896072.py:30: SyntaxWarning: invalid escape sequence '\p'
  clean_path = Path("D:\python projects\Mini Retail Sales Pipeline\data\processed")


In [49]:
# creating a fct table which hold all of the datasets

def build_sales_fact(orders, customers, products):
    fct_sales = orders_clean.merge(customers_clean, on="customer_id", how="inner")
    fct_sales = fct_sales.merge(products_clean, on="product_id", how="inner")

    fct_sales["month"] = fct_sales["order_date"].dt.to_period("M").dt.to_timestamp()
    fct_sales["gross_sales"] = fct_sales["quantity"] * fct_sales["unit_price"]
    fct_sales["discount_amount"] = fct_sales["gross_sales"] * fct_sales["discount_percent"] / 100
    fct_sales["net_sales"] = fct_sales["gross_sales"] - fct_sales["discount_amount"]
    fct_sales["total_cost"] = fct_sales["quantity"] * fct_sales["unit_cost"]
    fct_sales["profit"] = fct_sales["net_sales"] - fct_sales["total_cost"]
    fct_sales["profit_margin"] = fct_sales["profit"] / fct_sales["net_sales"]

    return fct_sales

fct_sales = build_sales_fact(orders_clean, customers_clean, products_clean)
fct_sales.to_csv(clean_path / "fct_sales.csv", index=False)


In [56]:
# creating a function for monthly KPIs
def create_monthly_kpis(fct_sales):
    monthly_kpis = (
        fct_sales[fct_sales["order_status"] == "Completed"]
        .groupby("month")
        .agg(
            total_orders=("order_id", "count"),
            total_customers=("customer_id", "nunique"),
            gross_sales=("gross_sales", "sum"),
            net_sales=("net_sales", "sum"),
            profit=("profit", "sum")
        ).reset_index())
    monthly_kpis["average_order_value"] = monthly_kpis["net_sales"] / monthly_kpis["total_orders"]
    return monthly_kpis

# creating a function for category kpis
def create_category_kpis(fct_sales):
    category_kpis = (
        fct_sales[fct_sales["order_status"] == "Completed"]
        .groupby("category").agg(
            total_orders=("order_id", "count"),
            total_quantity=("quantity", "sum"),
            net_sales=("net_sales", "sum"),
            total_profit=("profit", "sum"),
            profit_margin=("profit_margin", "sum")
        )
        .reset_index())
    return category_kpis

def create_customer_summary(fct_sales):
    customer_summary = (
        fct_sales[fct_sales["order_status"] == "Completed"]
        .groupby(["customer_id", "customer_name", "city"])
        .agg(
            total_orders=("order_id", "count"),
            total_spent=("net_sales", "sum"),
            total_profit=("profit", "sum")
        )
        .reset_index())

    customer_segments = []

    for total_spent in customer_summary["total_spent"]:
        if total_spent >= 1500:
            customer_segments.append("High Value")
        elif total_spent >= 800:
            customer_segments.append("Medium Value")
        else:
            customer_segments.append("Low Value")

    customer_summary["customer_segment"] = customer_segments

    return customer_summary



monthly_kpis = create_monthly_kpis(fct_sales)
category_kpis = create_category_kpis(fct_sales)
customer_summay = create_customer_summary(fct_sales)

monthly_kpis.to_csv(clean_path / "monthly_kpis.csv", index=False)
category_kpis.to_csv(clean_path / "category_kpis.csv", index=False)
customer_summay.to_csv(clean_path / "customer_summary.csv", index=False)

,order_id,order_date,customer_id,product_id,quantity,order_status,payment_method,discount_percent,customer_name,city,signup_date,product_name,category,unit_cost,unit_price
0,O1001,2026-01-03,C001,P001,2.0,Completed,Card,0.0,Mona Ali,Cairo,2025-11-02,Wireless Mouse,Accessories,180.0,300.0
1,O1002,2026-01-04,C002,P003,3.0,Completed,Cash,5.0,Omar Hassan,Alexandria,2025-11-15,USB-C Cable,Uncategorized,60.0,120.0
2,O1003,2026-01-06,C003,P006,1.0,Completed,Wallet,0.0,Nour Samir,Giza,2025-12-01,Headset,Electronics,350.0,650.0
3,O1004,2026-01-08,C004,P002,2.0,Cancelled,Card,0.0,Youssef Adel,Mansoura,2025-12-12,Keyboard,Accessories,250.0,450.0
4,O1005,2026-01-10,C005,P008,1.0,Completed,Card,10.0,Salma Nabil,Unknown,2026-01-04,Desk Lamp,Home Office,300.0,550.0
5,O1006,2026-01-13,C006,P004,2.0,Completed,Wallet,0.0,Karim Fathy,Cairo,2026-01-09,Laptop Stand,Accessories,220.0,400.0
6,O1007,2026-01-15,C007,P007,5.0,Completed,Cash,0.0,Laila Mostafa,Tanta,2026-01-16,Notebook Pack,Stationery,45.0,90.0
7,O1008,2026-01-17,C008,P005,1.0,Refunded,Card,0.0,Hana Sherif,Aswan,2026-01-22,Webcam,Electronics,500.0,850.0
8,O1009,2026-01-19,C009,P006,2.0,Completed,Wallet,5.0,Ahmed Tarek,Luxor,2026-02-03,Headset,Electronics,350.0,650.0
9,O1010,2026-01-20,C010,P003,4.0,Completed,Card,0.0,Farah Emad,Cairo,2026-02-10,USB-C Cable,Uncategorized,60.0,120.0
